In [1]:
from prompt_toolkit import keys
%matplotlib inline
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
from Utils.json_tools import read_formatted_json
from matplotlib import pyplot as plt
from scipy.ndimage import gaussian_filter1d
from plotting import plot_1d_rfmap

In [2]:
base_dir: Path = Path("/Volumes/senzailab/Kai/#Recording/m15/")
date = 260630

sessionID = 3
probe: str = "A"

session_dir = base_dir / str(date)
hd_session = session_dir / f"{date}_1"
rf_session = session_dir / f"{date}_{sessionID}"

In [3]:
rf_json_path = rf_session / f"data/rfmapping/good/0_200_1ms/Probe{probe}/regular_unitsSpikeCounts_{date}_{sessionID}.json"

rf_data = read_formatted_json(rf_json_path)
assert rf_data["unitsSpikeCountsSize"][3] == (len(rf_data['timeBinEdges']) - 1) == len(
    rf_data['unitsSpikeCounts'][0][0][0])
RFunitsSpikeCounts_time = np.asarray(rf_data['unitsSpikeCounts'])
RFunitPool = np.asarray(rf_data['unitPool'])

# Match the GUI's default RF sum range: snap 0 and 200 ms to the nearest bin edges.
RF_SUM_START_MS = 0.0
RF_SUM_END_MS = 200.0
RFtimeBinEdgesMs = np.asarray(rf_data['timeBinEdges'], dtype=float) * 1000.0
RFsumStartMs, RFsumEndMs = np.clip(
    [RF_SUM_START_MS, RF_SUM_END_MS],
    RFtimeBinEdgesMs[0],
    RFtimeBinEdgesMs[-1],
)
RFsumStartEdge = int(np.argmin(np.abs(RFtimeBinEdgesMs[:-1] - RFsumStartMs)))
RFsumEndEdge = int(np.argmin(np.abs(RFtimeBinEdgesMs[1:] - RFsumEndMs))) + 1
RFunitsSpikeCounts_sum_window = RFunitsSpikeCounts_time[..., RFsumStartEdge:RFsumEndEdge]

RFunitsSpikeCounts1D = RFunitsSpikeCounts_sum_window.sum(axis=(1, 3))
RFunitsSpikeCounts2D = RFunitsSpikeCounts_sum_window.sum(axis=3)
